# LIBERO: Benchmarking Knowledge Transfer for Lifelong Robot Learning

**Paper:** Liu, Zhu, Gao, Feng, Liu, Zhu, Stone — *LIBERO: Benchmarking Knowledge Transfer for Lifelong Robot Learning*, arXiv:2306.03310 (NeurIPS 2023, Datasets & Benchmarks track)

**Links:** [Paper](https://arxiv.org/abs/2306.03310) | [Project website](https://libero-project.github.io) | [Code](https://github.com/Lifelong-Robot-Learning/LIBERO) | [Docs](https://lifelong-robot-learning.github.io/LIBERO/)

---

### Table of contents

1. Theory — the LLDM problem formulation
2. Theory — BDDL and the procedural task-generation pipeline
3. Theory — the four task suites and what each one isolates
4. Theory — policy architectures (BC-RNN, BC-Transformer, BC-ViLT)
5. Theory — lifelong learning algorithms (SeqL, ER, EWC, PackNet, Multitask)
6. Theory — evaluation metrics (FWT, NBT, AUC) and where they come from
7. Headline empirical findings, with the caveats attached
8. Installation
9. Exploring the benchmark suites and tasks programmatically
10. Inspecting a BDDL problem file
11. Instantiating and stepping an environment; rendering observations
12. Visualizing a controlled distribution shift across a task suite
13. Loading and analyzing a human teleoperation demonstration
14. Configuring and launching an experiment (Hydra configs, `main.py`, `evaluate.py`)
15. Re-implementing the evaluation metrics from scratch, on toy data
16. LIBERO vs. CALVIN vs. RLBench — where this benchmark sits in the landscape
17. Discussion questions
18. References

### Why this benchmark, in one paragraph

Most "lifelong learning" benchmarks come from vision and NLP, where forgetting means forgetting *facts* — what a class of images looks like, what a word means. Robot manipulation policies have to remember something else too: *how to move*. LIBERO's whole design is built around teasing those two apart. Instead of one big pile of diverse tasks, it ships four suites, each holding almost everything about the task fixed except one axis of variation — object identity, spatial layout, or goal/behavior — so that when a lifelong learner forgets, you can tell *what* it forgot. That design choice is worth understanding in detail because it recurs across newer robot-learning benchmarks (and is exactly the kind of thing worth flagging when you're asked to summarize the paper's contribution in a meeting).

## 0. Notebook setup

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.style.use("dark_background")
mpl.rcParams.update({
    "figure.facecolor": "#1e1e1e",
    "axes.facecolor": "#1e1e1e",
    "savefig.facecolor": "#1e1e1e",
    "axes.edgecolor": "#888888",
    "axes.labelcolor": "#eeeeee",
    "xtick.color": "#cccccc",
    "ytick.color": "#cccccc",
    "text.color": "#eeeeee",
    "grid.color": "#3a3a3a",
    "font.size": 11,
    "figure.figsize": (7, 4),
})

SEED = 0
np.random.seed(SEED)
print("Setup complete.")

## 1. Theory: Lifelong Learning in Decision-Making (LLDM)

### 1.1 Why manipulation breaks the standard continual-learning setup

Continual/lifelong learning in vision and NLP is usually formalized as a sequence of *classification* problems: task $k$ supplies a distribution over $(x, y)$ pairs, and forgetting means the decision boundary for an earlier task's classes degrades. Everything that needs to transfer is **declarative** — facts about what things are.

A manipulation policy has to transfer a second kind of knowledge that classification tasks don't have: **procedural knowledge** — the actual motor skill of executing an action sequence (reach, grasp, insert, rotate...) to achieve a goal. Two tasks can share every object and every layout and still require completely different *behavior* (e.g. "open the drawer" vs. "close the drawer"), or share the identical behavior on completely different objects and layouts. Classification-style continual learning benchmarks have no way to represent that distinction, because there's only one axis (which class is correct) — LIBERO's whole design is about giving procedural knowledge its own axis.

### 1.2 Formal setup

A lifelong imitation-learning agent faces a **sequence** of $N$ tasks $\mathcal{T}_1, \dots, \mathcal{T}_N$, presented one at a time. Each task is a language-conditioned, sparse-reward MDP:

$$
\mathcal{T}_k = \langle \mathcal{S}, \mathcal{A}, P_k, R_k, \gamma, \ell_k \rangle
$$

- $\mathcal{S}$ — a **shared** observation space across all tasks: RGB images from an agent-view and a wrist/eye-in-hand camera, plus proprioceptive robot state (end-effector pose, gripper state, joint positions).
- $\mathcal{A} \subset \mathbb{R}^7$ — a **shared** end-effector delta-pose action space: 3-DoF translation, 3-DoF rotation (axis-angle), 1-DoF gripper open/close command.
- $P_k$ — task-specific transition dynamics, induced by the particular objects and layout of task $k$ (simulated in MuJoCo via `robosuite`).
- $R_k$ — **sparse**: $+1$ once the goal predicate in the task's BDDL file is satisfied, $0$ otherwise, every timestep until then.
- $\ell_k$ — a free-form natural-language instruction, e.g. *"put the black bowl in the bottom drawer of the cabinet and close it."*

Sharing $\mathcal{S}$ and $\mathcal{A}$ across every one of the 130 tasks is itself a deliberate design choice: it means a single network architecture can, in principle, be reused across the whole benchmark without any task-specific input/output surgery — so that when transfer succeeds or fails, it's attributable to the learning algorithm and not to an architecture mismatch.

### 1.3 The lifelong training protocol

For task $k$, the agent receives a demonstration dataset $\mathcal{D}_k = \{(s_i, a_i, \ell_k)\}_i$ of **human teleoperated** trajectories, trains on it (however its lifelong-learning algorithm dictates — with or without access to $\mathcal{D}_1, \dots, \mathcal{D}_{k-1}$), and then moves to task $k+1$ — it cannot go back and re-train jointly on old data unless its algorithm explicitly maintains a replay buffer. After each task $k$, the policy is evaluated by rollout (not by matching actions to the dataset) on **every task seen so far**, $1$ through $k$, using the environment's sparse success signal. This is exactly the information needed to build the success matrix $S_{k,j}$ we'll use in Section 15.

## 2. Theory: BDDL and the procedural generation pipeline

Every LIBERO task is specified by a **BDDL** (Behavior Domain Definition Language, inherited and adapted from the BEHAVIOR benchmark) file — a symbolic, PDDL-like description that separates *what a scene contains* from *what MuJoCo XML it compiles to*. A BDDL file declares:

- **Regions** — named areas of the table/scene (e.g. `plate_region`, `drawer_top_region`) used both to procedurally place objects and to define goal predicates.
- **Fixtures and objects** — which asset (drawer, bowl, plate, cabinet, ...) goes where, drawn from a library of textured, physically-annotated MuJoCo assets.
- **Initial-state predicates** — logical statements like `(On bowl_1 plate_region)` that a scene sampler satisfies when instantiating an episode (this is also what lets `get_task_init_states` return a *fixed, reproducible* pool of valid starting configurations).
- **Goal predicates** — the logical condition that must hold for the sparse reward to fire, e.g. `(And (On bowl_1 plate_1) (Open drawer_1))`.

A simplified illustrative excerpt (not copied from the repo, illustrative of the structure) looks like:

```
(define (problem LIBERO_Spatial_Task)
  (:domain robosuite)
  (:objects
    akita_black_bowl_1 - object
    plate_1            - object
    ramekin_1          - object)
  (:regions
    (plate_region ...)
    (target_region ...))
  (:init
    (On akita_black_bowl_1 table)
    (On plate_1 table)
    (On ramekin_1 table))
  (:goal
    (And (On akita_black_bowl_1 plate_1))))
```

**Why this matters for the "extendible" claim in the abstract:** because the goal and initial-state logic are declared symbolically rather than hard-coded per-scene, the same generation pipeline that produced the 130 shipped tasks can sample *new* object sets, layouts, and goal predicates — the paper's claim that the pipeline can "in principle generate infinitely many tasks" refers to this decoupling of symbolic task specification from physical scene instantiation, not to 130 being an exhaustive set.

## 3. Theory: the four task suites, in detail

| Suite | # tasks | What's held fixed | What varies | Knowledge isolated |
|---|---|---|---|---|
| **LIBERO-Spatial** | 10 | Object set, task goal | Spatial arrangement of objects | Declarative — spatial relationships |
| **LIBERO-Object** | 10 | Layout, task goal/behavior | Which object is manipulated | Declarative — object identity |
| **LIBERO-Goal** | 10 | Object set, layout | The goal / required behavior | Procedural — motor behaviors |
| **LIBERO-100** | 100 | — (least controlled) | Everything: objects, layout, goals, "long-horizon" multi-step goals | Entangled declarative + procedural |

**LIBERO-100** is further split by role, not by content:
- **LIBERO-90** — 90 tasks intended purely as a large-scale **pretraining** corpus (e.g. for representation learning or multitask pretraining before the actual lifelong-learning evaluation).
- **LIBERO-10** (also referred to as **LIBERO-Long** in places, since its tasks tend to be longer-horizon, multi-stage goals) — the held-out 10-task **downstream lifelong-learning** evaluation suite.

So a typical experiment in the paper either (a) runs the lifelong protocol directly on LIBERO-Spatial / -Object / -Goal to isolate one knowledge type, or (b) pretrains on LIBERO-90 and then runs the lifelong protocol on LIBERO-10 to study transfer from broad pretraining into a genuinely lifelong, longer-horizon setting.

**A design point worth internalizing:** within LIBERO-Spatial/-Object/-Goal, task *order* is itself an experimental variable (research question 4 in the abstract — robustness to task ordering) — because everything else is controlled, differences in FWT/NBT across orderings can be attributed to the *order* rather than to confounded task difficulty.

## 4. Theory: policy architectures under study

LIBERO benchmarks three flat (non-hierarchical) vision-language-conditioned behavior-cloning architectures, all sharing the same high-level recipe: encode each camera image, fuse with proprioception and a language/task embedding, integrate over time, and regress an action. They differ in **how images are encoded** and **how temporal context is integrated**:

| Name in repo | Shorthand | Visual encoder | Temporal integration |
|---|---|---|---|
| `bc_rnn_policy` | **ResNet-RNN** | ResNet (CNN) | LSTM/GRU recurrence over the trajectory |
| `bc_transformer_policy` | **ResNet-T** | ResNet (CNN) | Causal transformer over a window of past frames |
| `bc_vilt_policy` | **ViT-T** | Vision Transformer (patch-based) | Causal transformer over a window of past frames |

All three condition on the language instruction $\ell_k$ through a pretrained sentence embedding (the paper compares BERT, CLIP-text, GPT-2, and a simple learned Task-ID embedding as the source of that embedding — finding, somewhat surprisingly, that a bag-of-words-level Task-ID embedding is competitive with much larger pretrained language models, which the authors read as evidence that current fusion mechanisms aren't yet extracting much *semantic* content from the instruction beyond "which task is this").

A minimal PyTorch sketch of the shared skeleton (not the exact repo implementation, but architecturally faithful) — useful for building intuition about where "declarative" and "procedural" information actually enters the network:

In [ ]:
import torch
import torch.nn as nn


class VisualEncoder(nn.Module):
    """Stand-in for either a small ResNet (ResNet-RNN / ResNet-T) or a ViT patch
    encoder (ViT-T). Maps a (B, T, C, H, W) stack of RGB frames to per-frame embeddings.
    """

    def __init__(self, embed_dim: int = 128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(64, embed_dim)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = images.shape
        feats = self.backbone(images.view(B * T, C, H, W)).flatten(1)
        return self.proj(feats).view(B, T, -1)


class LIBEROPolicySkeleton(nn.Module):
    """Shared skeleton of the three LIBERO policy architectures: encode vision,
    fuse with proprioception + a language/task embedding, integrate over time,
    regress a 7-DoF action. `temporal` selects LSTM (ResNet-RNN) vs. causal
    self-attention (ResNet-T / ViT-T) integration.
    """

    def __init__(self, embed_dim: int = 128, proprio_dim: int = 9,
                 lang_dim: int = 768, temporal: str = "transformer"):
        super().__init__()
        self.vision = VisualEncoder(embed_dim)
        self.proprio_proj = nn.Linear(proprio_dim, embed_dim)
        self.lang_proj = nn.Linear(lang_dim, embed_dim)  # BERT/CLIP/GPT-2/Task-ID embedding in
        fused_dim = embed_dim  # after summing vision + proprio + language tokens

        if temporal == "lstm":
            self.temporal_model = nn.LSTM(fused_dim, fused_dim, batch_first=True)
        else:
            encoder_layer = nn.TransformerEncoderLayer(fused_dim, nhead=4, batch_first=True)
            self.temporal_model = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.action_head = nn.Linear(fused_dim, 7)  # 3 translation + 3 rotation + 1 gripper

    def forward(self, images, proprio, lang_embed):
        vis_feat = self.vision(images)                      # (B, T, D)
        proprio_feat = self.proprio_proj(proprio)            # (B, T, D)
        lang_feat = self.lang_proj(lang_embed)[:, None, :]   # (B, 1, D), broadcast over T
        fused = vis_feat + proprio_feat + lang_feat

        if isinstance(self.temporal_model, nn.LSTM):
            temporal_out, _ = self.temporal_model(fused)
        else:
            # causal mask so the transformer can't attend to the future
            T = fused.shape[1]
            causal_mask = torch.triu(torch.full((T, T), float("-inf")), diagonal=1)
            temporal_out = self.temporal_model(fused, mask=causal_mask)

        return self.action_head(temporal_out)  # (B, T, 7) predicted action per timestep


# sanity-check the skeleton on random data
model = LIBEROPolicySkeleton(temporal="transformer")
B, T, H, W = 2, 5, 84, 84
dummy_images = torch.randn(B, T, 3, H, W)
dummy_proprio = torch.randn(B, T, 9)
dummy_lang = torch.randn(B, 768)
out = model(dummy_images, dummy_proprio, dummy_lang)
print("predicted action sequence shape:", tuple(out.shape))  # (B, T, 7)

## 5. Theory: lifelong learning algorithms under study

The `libero/lifelong/algos/` module implements a common base class (`Sequential`) that every algorithm below subclasses — the differences are all in what extra machinery wraps the plain sequential-fine-tuning loop.

### 5.1 Sequential fine-tuning (SeqL) — the baseline

No anti-forgetting mechanism at all: train on $\mathcal{D}_k$, minimize behavior-cloning loss $\mathcal{L}_{BC} = \mathbb{E}_{(s,a,\ell_k) \sim \mathcal{D}_k}\big[\|\pi_\theta(s, \ell_k) - a\|^2\big]$, move on. It is the effective *upper bound* on forward transfer (nothing constrains adaptation to the new task) and, in principle, the worst case for backward transfer.

### 5.2 Experience Replay (ER)

Maintains a fixed-size buffer $\mathcal{M}$ sampled from past tasks' data. At each training step on task $k$, the loss is computed over a mixed batch from $\mathcal{D}_k \cup \mathcal{M}$, and $\mathcal{M}$ is updated (e.g. reservoir sampling) after task $k$ finishes. It directly rehearses old behavior at the cost of memory and mixing old/new gradient signal.

### 5.3 Elastic Weight Consolidation (EWC)

A regularization-based approach. After finishing task $k-1$, EWC estimates a **Fisher information matrix** $F_{k-1}$ over parameters $\theta$ — intuitively, how sensitive task $k-1$'s loss is to perturbing each parameter. When training on task $k$, it adds a quadratic penalty that discourages moving parameters that mattered for earlier tasks:

$$
\mathcal{L}_k(\theta) = \mathcal{L}_{BC}(\theta; \mathcal{D}_k) + \frac{\lambda}{2} \sum_{i} F_{k-1,i}\,(\theta_i - \theta^{*}_{k-1,i})^2
$$

where $\theta^{*}_{k-1}$ are the parameters at the end of task $k-1$ and $\lambda$ trades off plasticity vs. stability. The paper's finding that EWC underperforms plain SeqL is a useful, slightly counter-intuitive result to know going in: an explicit anti-forgetting regularizer can *reduce* the network's ability to adapt at all, and in LLDM (where each new task can require substantially different procedural behavior, not just a shifted decision boundary) that plasticity loss can outweigh the forgetting it prevents.

### 5.4 PackNet

A **parameter-isolation** approach: after training on task $k$, PackNet iteratively prunes the network and *freezes* the surviving weights as task $k$'s dedicated subnetwork, leaving the pruned capacity free for future tasks. This guarantees zero forgetting on frozen weights by construction, but the total network capacity is finite and gets exhausted — the paper's finding that PackNet struggles specifically on LIBERO-10 (the longer-horizon suite) is attributed to those tasks needing more capacity per task than PackNet's pruning schedule leaves available.

### 5.5 Multitask learning (MTL) — an oracle-ish baseline, not a lifelong algorithm

Trains on $\bigcup_k \mathcal{D}_k$ jointly from the start rather than sequentially — no lifelong constraint at all. It's included as an upper-bound reference for "what if you could just have all the data at once," not as a method the field is proposing.

A minimal, architecture-agnostic sketch of the EWC penalty (the piece that's easiest to get subtly wrong) — this is illustrative math, not the repo's exact implementation:

In [ ]:
def ewc_penalty(model: "torch.nn.Module",
                 fisher: dict,
                 theta_star: dict,
                 lam: float = 100.0) -> "torch.Tensor":
    """Quadratic EWC penalty: (lambda / 2) * sum_i F_i * (theta_i - theta*_i)^2.

    Parameters
    ----------
    model : torch.nn.Module
        The current policy being trained on task k.
    fisher : dict[str, torch.Tensor]
        Per-parameter Fisher information estimated at the end of task k-1
        (diagonal approximation — one scalar per parameter element).
    theta_star : dict[str, torch.Tensor]
        A snapshot of `model`'s parameters at the end of task k-1.
    lam : float
        Regularization strength (stability vs. plasticity trade-off).
    """
    penalty = 0.0
    for name, param in model.named_parameters():
        if name in fisher:
            penalty = penalty + (fisher[name] * (param - theta_star[name]).pow(2)).sum()
    return 0.5 * lam * penalty


def estimate_diagonal_fisher(model, dataloader, loss_fn, n_batches: int = 50) -> dict:
    """Diagonal Fisher information estimate: average squared gradient of the
    (behavior-cloning) loss w.r.t. each parameter, over a sample of task k-1's data.
    Called once, right after finishing training on a task, before moving to the next.
    """
    fisher = {name: torch.zeros_like(p) for name, p in model.named_parameters()}
    model.eval()
    for i, batch in enumerate(dataloader):
        if i >= n_batches:
            break
        model.zero_grad()
        loss = loss_fn(model, batch)
        loss.backward()
        for name, p in model.named_parameters():
            if p.grad is not None:
                fisher[name] += p.grad.detach() ** 2
    for name in fisher:
        fisher[name] /= min(n_batches, i + 1)
    return fisher

print("EWC penalty + diagonal-Fisher estimator defined (illustrative, not the exact repo code).")

## 6. Theory: evaluation metrics, and where they come from

LIBERO reports success rate as $s_{k,j} \in [0,1]$: the fraction of evaluation rollouts on task $j$ that succeed, measured right after the agent has *finished sequential training through task $k$* (so $j \le k$; $s_{k,j}$ for $j > k$ is undefined — the agent hasn't seen task $j$ yet). This success matrix is the common currency the three headline metrics are computed from. The metric *names* (forward transfer, backward transfer) come from the continual-learning literature (notably Lopez-Paz & Ranzato's Gradient Episodic Memory paper), adapted here to a rollout-success rather than classification-accuracy signal.

**Forward transfer (FWT).** Does prior task exposure make the agent *better* at a new task than training on it in isolation would?

$$
\mathrm{FWT}_k = s_{k,k} - s_{k,k}^{\text{scratch}}, \qquad \mathrm{FWT} = \frac{1}{N}\sum_{k=1}^N \mathrm{FWT}_k
$$

where $s_{k,k}^{\text{scratch}}$ is the success rate of a policy trained on task $k$ **alone** (the single-task baseline), decoupled from the lifelong sequence. FWT can be positive (prior tasks helped — e.g. shared grasping skill) or negative (prior tasks actively hurt — e.g. interference between conflicting behaviors).

**Negative backward transfer (NBT), i.e. forgetting.** How much does an *old* task's performance degrade by the time the agent finishes the whole sequence?

$$
\mathrm{NBT} = \frac{1}{N-1}\sum_{j=1}^{N-1} \big(s_{j,j} - s_{N,j}\big)
$$

Positive NBT = the agent forgot task $j$ after learning later tasks (catastrophic forgetting); negative NBT would mean later tasks *improved* an earlier one (positive backward transfer) — rare, but not impossible if tasks share reusable sub-skills.

**Area under the success-rate curve (AUC).** A single scalar that rewards both forward transfer and forgetting-resistance jointly, by averaging success over *all tasks seen so far* at every point in the sequence:

$$
\mathrm{AUC} = \frac{1}{N}\sum_{k=1}^{N}\left(\frac{1}{k}\sum_{j=1}^{k} s_{k,j}\right)
$$

**Why three metrics and not one.** A method can win on FWT by aggressively overwriting old knowledge (bad NBT), or win on NBT by barely adapting at all (bad FWT, e.g. a frozen network trivially has zero forgetting). AUC is the metric to report if you can only report one, but FWT and NBT are what let you *diagnose why* a method got the AUC it did — which is exactly why the paper's headline claim ("SeqL beats purpose-built lifelong methods") is specifically a claim about FWT, not about AUC or NBT.

## 7. Headline empirical findings — with the caveats attached

It's easy to over-summarize these into soundbites, so here they are with the qualifier that actually matters for each:

1. **Sequential fine-tuning (SeqL) beats ER, EWC, and PackNet on forward transfer**, given sufficient model capacity — *but* this is specifically a forward-transfer result. SeqL has no mechanism to resist forgetting, so this finding does not say "just use SeqL and ignore lifelong learning" — it says purpose-built anti-forgetting mechanisms currently trade away plasticity faster than they buy back forgetting-resistance, at least at the model scales studied.
2. **No single visual encoder wins universally.** ResNet-T and ViT-T (both transformer-temporal) clearly beat ResNet-RNN on average. Between ResNet-T and ViT-T, the ranking *flips* depending on the lifelong algorithm: under PackNet, they're statistically indistinguishable except on the longer-horizon LIBERO-10/LIBERO-Long suite (where ViT-T wins); under ER, ResNet-T wins on most suites except LIBERO-Object. The takeaway is that "best architecture" is not a property of the architecture alone — it interacts with the lifelong algorithm and the specific knowledge-transfer axis being tested.
3. **Pretrained language embeddings (BERT/CLIP/GPT-2) don't reliably beat a simple learned Task-ID embedding.** The authors' reading is that the current fusion mechanisms are mostly using the instruction as a bag-of-words task discriminator rather than extracting compositional semantic content — a concrete, testable hypothesis if you're thinking about improving instruction-following in a follow-up project.
4. **Naive supervised pretraining on LIBERO-90 can hurt subsequent lifelong learning on LIBERO-10.** This is the most counter-intuitive finding and worth treating carefully: it's a claim about *naive* pretraining under the specific setup studied, not a blanket claim that pretraining never helps robot policies (a large and growing body of VLA work post-dates this paper and pretrains successfully at much larger scale) — but within this benchmark's controlled setting, it's a real negative result and a reminder to actually measure downstream lifelong performance rather than assuming pretraining monotonically helps.

If you're citing this paper in a meeting, findings (1) and (4) are the two most likely to get pushback/questions, since both cut against a naive intuition ("purpose-built methods should beat the naive baseline"; "more pretraining should help") — worth having the caveat above ready.

## 8. Installation

LIBERO is built on [`robosuite`](https://robosuite.ai/) (which wraps MuJoCo) plus `bddl` for parsing task definition files. The cell below mirrors the official install instructions, configured for a **headless/CPU runtime** (Colab, a lab server without a display) via the EGL rendering backend.

> Heavy cell (MuJoCo + robosuite + LIBERO from source) — commented out by default so `Run All` doesn't trigger a multi-minute install. Uncomment to actually install; restart the runtime afterward.

In [ ]:
# %%bash
# apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libosmesa6 libglfw3 patchelf > /dev/null
# pip install -q robosuite bddl easydict hydra-core cloudpickle h5py
# git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git
# pip install -q -e LIBERO

import os
os.environ.setdefault("MUJOCO_GL", "egl")  # headless rendering backend

try:
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv
    LIBERO_AVAILABLE = True
    print("libero import OK")
except ImportError:
    LIBERO_AVAILABLE = False
    print("libero is not installed in this environment — uncomment the install cell above and restart "
          "the runtime to run the live simulation cells below. Everything else (theory, metrics, "
          "architecture/algorithm sketches) runs regardless.")

## 9. Exploring the benchmark suites and tasks programmatically

Every task suite is registered in a `benchmark_dict`. Let's list every suite and then print **every task's language instruction** within one suite — this is the fastest way to build intuition for what "controlled distribution shift" actually looks like in practice, rather than just reading the definition.

In [ ]:
if LIBERO_AVAILABLE:
    benchmark_dict = benchmark.get_benchmark_dict()
    print("Registered task suites:")
    for suite_name in benchmark_dict:
        suite = benchmark_dict[suite_name]()
        print(f"  {suite_name:16s}  {suite.n_tasks:3d} tasks")
else:
    reference_suite_sizes = {
        "libero_spatial": 10, "libero_object": 10, "libero_goal": 10,
        "libero_90": 90, "libero_10": 10,
    }
    print("Registered task suites (reference values):")
    for name, n in reference_suite_sizes.items():
        print(f"  {name:16s}  {n:3d} tasks")

In [ ]:
if LIBERO_AVAILABLE:
    task_suite_name = "libero_spatial"
    task_suite = benchmark_dict[task_suite_name]()

    print(f"All {task_suite.n_tasks} language instructions in {task_suite_name} "
          f"(same objects/goal, spatial layout varies):\n")
    for tid in range(task_suite.n_tasks):
        task = task_suite.get_task(tid)
        print(f"  [{tid}] {task.language}")
else:
    print('Example language instructions from LIBERO-Spatial (same object set and goal type,\n'
          'only the spatial arrangement varies between tasks):\n')
    examples = [
        "pick up the black bowl between the plate and the ramekin and place it on the plate",
        "pick up the black bowl next to the ramekin and place it on the plate",
        "pick up the black bowl on the stove and place it on the plate",
        "pick up the black bowl on the cookie box and place it on the plate",
    ]
    for i, ex in enumerate(examples):
        print(f"  [{i}] {ex}")
    print("  ... (10 total; notice the goal — place the bowl on the plate — never changes, only where\n"
          "      the bowl starts relative to other objects does)")

## 10. Inspecting a BDDL problem file

Each task's `task_bddl_file` path points at the symbolic problem definition introduced in Section 2. Let's actually read one and look at its structure, rather than taking the theory section's illustrative excerpt on faith.

In [ ]:
if LIBERO_AVAILABLE:
    task_id = 0
    task = task_suite.get_task(task_id)
    task_bddl_file = os.path.join(
        get_libero_path("bddl_files"), task.problem_folder, task.bddl_file
    )
    print(f"task name : {task.name}")
    print(f"language  : \"{task.language}\"")
    print(f"bddl file : {task_bddl_file}\n")

    with open(task_bddl_file) as f:
        bddl_text = f.read()
    print(bddl_text[:1500])
    if len(bddl_text) > 1500:
        print("... (truncated)")
else:
    print("Reading a real .bddl file requires the live install (Section 8). Structurally, expect to see:\n"
          "  (:objects ...)   — which asset instances populate the scene\n"
          "  (:regions ...)   — named placement/goal regions on the table\n"
          "  (:init ...)      — predicates the scene sampler must satisfy when instantiating an episode\n"
          "  (:goal ...)      — the predicate that triggers the sparse +1 reward")

## 11. Instantiating and stepping an environment; rendering observations

`OffScreenRenderEnv` wraps `robosuite`'s MuJoCo simulation and renders RGB camera observations without a display — the right choice in a notebook. Task suites ship **fixed initial states** via `get_task_init_states`, so every algorithm evaluated on LIBERO is benchmarked from an identical pool of starting configurations, which is what makes success rates comparable across papers.

In [ ]:
if LIBERO_AVAILABLE:
    env_args = {
        "bddl_file_name": task_bddl_file,
        "camera_heights": 256,
        "camera_widths": 256,
    }
    env = OffScreenRenderEnv(**env_args)
    env.seed(SEED)
    env.reset()

    init_states = task_suite.get_task_init_states(task_id)
    print(f"{len(init_states)} fixed initial states available for this task")
    env.set_init_state(init_states[0])

    dummy_action = [0.0] * 7  # hold still
    frames, gripper_qpos = [], []
    for step in range(20):
        obs, reward, done, info = env.step(dummy_action)
        frames.append(obs["agentview_image"])
        gripper_qpos.append(obs.get("robot0_gripper_qpos", np.zeros(2)))
    env.close()

    fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
    for ax, idx in zip(axes, [0, 6, 13, 19]):
        ax.imshow(frames[idx])
        ax.set_title(f"step {idx}")
        ax.axis("off")
    fig.suptitle(f'"{task.language}"', y=1.03)
    plt.tight_layout()
    plt.show()
else:
    print("Environment stepping requires a live `libero` install (Section 8). With it, this cell renders\n"
          "agentview RGB frames over 20 steps of a zero (hold-still) action, confirming the scene matches\n"
          "the language instruction retrieved above.")

## 12. Visualizing the controlled distribution shift across a suite

The theory section claimed LIBERO-Spatial holds objects/goal fixed and varies only spatial layout. Let's *see* that directly by rendering the initial frame of several tasks in the suite side by side — this is the kind of sanity check worth doing before trusting a benchmark's own description of itself, and it's a good habit to bring into any paper reproduction.

In [ ]:
if LIBERO_AVAILABLE:
    n_show = min(4, task_suite.n_tasks)
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 3.6))
    for i in range(n_show):
        t = task_suite.get_task(i)
        bddl_path = os.path.join(get_libero_path("bddl_files"), t.problem_folder, t.bddl_file)
        e = OffScreenRenderEnv(bddl_file_name=bddl_path, camera_heights=256, camera_widths=256)
        e.seed(SEED)
        e.reset()
        e.set_init_state(task_suite.get_task_init_states(i)[0])
        obs, *_ = e.step([0.0] * 7)
        e.close()

        ax = axes[i] if n_show > 1 else axes
        ax.imshow(obs["agentview_image"])
        ax.set_title(t.language, fontsize=8, wrap=True)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("With a live install, this cell tiles the initial agentview frame of several LIBERO-Spatial\n"
          "tasks side by side — visually, the bowl/plate/ramekin/stove assets and the final goal look the\n"
          "same across panels, and only the object positions relative to each other differ. That's the\n"
          "'declarative — spatial relationships' shift made concrete.")

## 13. Loading and analyzing a human teleoperation demonstration

LIBERO ships high-quality **human teleoperated** demonstrations for every task (downloaded via `benchmark_scripts/download_libero_datasets.py`), stored in the `robomimic` HDF5 convention: one group per demo under `data/`, each holding `obs/` (per-modality observation streams), `actions`, `states` (full simulator state, for exact replay), and `rewards`. This is exactly what a behavior-cloning policy — any of the three architectures from Section 4 — trains on.

In [ ]:
import h5py


def inspect_demo_file(hdf5_path: str) -> None:
    """Print the structure of a LIBERO/robomimic-format demonstration file:
    number of demos, per-demo length, and available observation modalities.

    Parameters
    ----------
    hdf5_path : str
        Path to a `<task_name>_demo.hdf5` file produced by LIBERO's dataset download script.
    """
    with h5py.File(hdf5_path, "r") as f:
        demo_keys = list(f["data"].keys())
        print(f"{len(demo_keys)} demonstrations in {os.path.basename(hdf5_path)}")

        demo0 = f["data"][demo_keys[0]]
        actions = demo0["actions"][()]
        print(f"first demo length       : {actions.shape[0]} timesteps")
        print(f"action dimensionality   : {actions.shape[1]} (3 translation + 3 rotation + 1 gripper)")
        print(f"observation modalities  : {list(demo0['obs'].keys())}")
        lengths = [f['data'][k]['actions'].shape[0] for k in demo_keys]
        print(f"trajectory length stats : min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}")


def plot_demo_trajectory(hdf5_path: str, demo_idx: int = 0) -> None:
    """Plot the end-effector position trajectory (if available) and the gripper
    action over time for one demonstration, as a sanity check on the data before
    it goes into a training pipeline.
    """
    with h5py.File(hdf5_path, "r") as f:
        demo_keys = list(f["data"].keys())
        demo = f["data"][demo_keys[demo_idx]]
        actions = demo["actions"][()]

        fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

        if "robot0_eef_pos" in demo["obs"]:
            eef_pos = demo["obs"]["robot0_eef_pos"][()]
            axes[0].plot(eef_pos[:, 0], label="x", color="#ef5350")
            axes[0].plot(eef_pos[:, 1], label="y", color="#66bb6a")
            axes[0].plot(eef_pos[:, 2], label="z", color="#42a5f5")
            axes[0].set_title("end-effector position")
            axes[0].set_xlabel("timestep")
            axes[0].legend()
        else:
            axes[0].text(0.5, 0.5, "robot0_eef_pos not in obs", ha="center", va="center")
            axes[0].axis("off")

        axes[1].plot(actions[:, -1], color="#ffca28")
        axes[1].set_title("gripper action (dim 6): +1 close / -1 open")
        axes[1].set_xlabel("timestep")

        plt.suptitle(f"{demo_keys[demo_idx]}  ({actions.shape[0]} steps)")
        plt.tight_layout()
        plt.show()


demo_path = "libero_spatial/pick_up_the_black_bowl_..._demo.hdf5"  # fill in after running the downloader
if LIBERO_AVAILABLE and os.path.exists(demo_path):
    inspect_demo_file(demo_path)
    plot_demo_trajectory(demo_path)
else:
    print("No local demo file found. Run:\n"
          "  python benchmark_scripts/download_libero_datasets.py --datasets libero_spatial\n"
          "from the LIBERO repo root, then point `demo_path` at one of the resulting .hdf5 files.\n\n"
          "Once available, `inspect_demo_file` prints demo counts/lengths/modalities and\n"
          "`plot_demo_trajectory` plots the end-effector path and gripper open/close signal —\n"
          "useful both for a sanity check and for building intuition about what the action\n"
          "space in Section 1.2 actually looks like on real teleoperated data.")

## 14. Configuring and launching a lifelong-learning experiment

Training/evaluation is driven by [Hydra](https://hydra.cc/) configs under `libero/configs/`, composed from independent groups:

```
configs/
  config.yaml               # top-level config that composes everything below
  data/default.yaml          # dataset paths, batch size, augmentation
  eval/default.yaml           # rollout count, eval frequency
  lifelong/
    base.yaml                # sequential fine-tuning (no extra mechanism)
    er.yaml  ewc.yaml  packnet.yaml  multitask.yaml  single_task.yaml
  policy/
    bc_rnn_policy.yaml        # ResNet-LSTM
    bc_transformer_policy.yaml  # ResNet-Transformer
    bc_vilt_policy.yaml       # ViT-Transformer
  train/default.yaml          # optimizer, scheduler, epochs
```

A full experiment — reproducing, say, "ResNet-T + ER on LIBERO-Spatial" — is launched with:

```bash
export CUDA_VISIBLE_DEVICES=0
export MUJOCO_EGL_DEVICE_ID=0
python libero/lifelong/main.py seed=0 \
                                benchmark_name=libero_spatial \
                                policy=bc_transformer_policy \
                                lifelong=er
```

**What `main.py` does, conceptually** (this is the loop the theory sections above are describing in code form):

```python
# pseudocode, mirroring libero/lifelong/main.py at a high level
benchmark = benchmark_dict[cfg.benchmark_name]()
algo = get_algo_class(cfg.lifelong)(policy_cfg=cfg.policy)  # SeqL / ER / EWC / PackNet / MTL

success_matrix = np.full((benchmark.n_tasks, benchmark.n_tasks), np.nan)

for k in range(benchmark.n_tasks):
    dataset_k = load_demo_dataset(benchmark, task_id=k)
    algo.learn_one_task(dataset_k, task_id=k)     # this is where ER/EWC/PackNet differ from SeqL

    for j in range(k + 1):                        # evaluate on every task seen so far
        env_j = make_env(benchmark, task_id=j)
        success_matrix[k, j] = rollout_and_evaluate(algo.policy, env_j, benchmark.get_task_init_states(j))

# success_matrix is exactly the S_{k,j} used to compute FWT / NBT / AUC in Section 6
```

If GPU budget is limited, you don't have to evaluate on the fly — `libero/lifelong/evaluate.py` re-runs rollout evaluation from a saved checkpoint after the fact:

```bash
python libero/lifelong/evaluate.py --benchmark libero_spatial \
                                    --task_id 3 --algo er --policy bc_transformer_policy \
                                    --seed 0 --ep 50 --load_task 9 --device_id 0
```

`--load_task 9` here means "load the checkpoint saved after training on task index 9 (i.e. the end of the full sequence)," and `--task_id 3` selects which task's success rate to measure — exactly one entry $S_{9,3}$ of the success matrix.

## 15. Re-implementing the evaluation metrics from scratch, on toy data

Before wiring FWT/NBT/AUC into a real training loop, let's pin the definitions down on synthetic data. We model a success-rate matrix $S \in [0,1]^{N\times N}$ where $S[k,j] = s_{k,j}$ — success on task $j$ measured right after sequential training through task $k$ (entries with $j > k$ are undefined and left as `NaN`). This is exactly the `success_matrix` produced by the `main.py` pseudocode above.

In [ ]:
def forward_transfer(S: np.ndarray, S_scratch_diag: np.ndarray) -> np.ndarray:
    """Per-task forward transfer FWT_k = s_{k,k} - s_{k,k}^scratch.

    Parameters
    ----------
    S : (N, N) array
        Lower-triangular success matrix; S[k, j] is success on task j after
        sequential training through task k (j <= k).
    S_scratch_diag : (N,) array
        Success rate on task k when trained on task k alone (single-task baseline),
        i.e. s_{k,k}^scratch for each k.

    Returns
    -------
    (N,) array of per-task forward transfer.
    """
    diag = np.diag(S)
    return diag - S_scratch_diag


def negative_backward_transfer(S: np.ndarray) -> float:
    """NBT = mean over tasks j < N of (s_{j,j} - s_{N,j}): how much each task's
    performance dropped by the end of the sequence relative to right after it was learned.
    Positive NBT = forgetting; negative NBT = later tasks improved earlier ones.
    """
    diag = np.diag(S)[:-1]          # s_{j,j} for j = 0..N-2
    final_row = S[-1, :-1]          # s_{N,j} for j = 0..N-2
    return float(np.mean(diag - final_row))


def success_curve_auc(S: np.ndarray) -> float:
    """AUC = mean over k of the average success across tasks 1..k after training on task k."""
    N = S.shape[0]
    per_k_avg = [np.nanmean(S[k, : k + 1]) for k in range(N)]
    return float(np.mean(per_k_avg))


# --- toy example: 5-task sequence ---
N = 5
rng = np.random.default_rng(SEED)
S = np.full((N, N), np.nan)
for k in range(N):
    for j in range(k + 1):
        age = k - j  # how many tasks ago was j learned, relative to now
        S[k, j] = np.clip(0.85 - 0.06 * age + rng.normal(0, 0.03), 0, 1)

S_scratch_diag = np.clip(np.diag(S) - rng.uniform(0.05, 0.15, size=N), 0, 1)  # scratch is a bit worse

ft = forward_transfer(S, S_scratch_diag)
nbt = negative_backward_transfer(S)
auc = success_curve_auc(S)

print("Success matrix S (rows = after training on task k, cols = evaluated on task j):")
print(np.round(S, 2))
print(f"\nForward transfer per task : {np.round(ft, 3)}")
print(f"Negative backward transfer: {nbt:.3f}  (higher = more forgetting)")
print(f"AUC                       : {auc:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im = axes[0].imshow(S, cmap="viridis", vmin=0, vmax=1)
axes[0].set_xlabel("task j (evaluated on)")
axes[0].set_ylabel("after training on task k")
axes[0].set_title("Success matrix $S_{k,j}$")
fig.colorbar(im, ax=axes[0], fraction=0.046)

per_k_avg = [np.nanmean(S[k, : k + 1]) for k in range(N)]
axes[1].plot(range(1, N + 1), per_k_avg, "o-", color="#ffca28")
axes[1].axhline(auc, color="#ef5350", linestyle="--", label=f"AUC = {auc:.3f}")
axes[1].set_xlabel("tasks seen so far (k)")
axes[1].set_ylabel("avg. success across tasks 1..k")
axes[1].set_title("Running average success curve")
axes[1].legend()

plt.tight_layout()
plt.show()

**A second toy scenario worth running yourself:** set `age` in the loop above to `0.0 * age` (i.e. remove the forgetting term entirely) and recompute — you should get `NBT ≈ 0`. Then try making later tasks *increase* earlier ones' success (`+0.03 * age` instead of `-0.06 * age`) and confirm NBT goes negative. Being able to predict the sign of NBT from a scenario description, and vice versa, is a good check that the definition has actually landed.

## 16. LIBERO vs. CALVIN vs. RLBench — where this benchmark sits in the landscape

Since you've already built tutorials on CALVIN and RLBench for `xulabs/edu`, it's worth placing LIBERO relative to both rather than treating it as a fourth unrelated benchmark:

| | **LIBERO** | **CALVIN** | **RLBench** |
|---|---|---|---|
| Primary axis studied | Lifelong learning: declarative vs. procedural knowledge transfer, isolated per suite | Long-horizon, language-conditioned skill chaining in one continuous environment | Broad single-task and multi-task manipulation skill breadth |
| Task structure | 130 discrete tasks across 4 suites, each a single language-conditioned goal | A small set of atomic skills chained into long instruction sequences in one shared scene | 100+ hand-designed tasks, typically evaluated independently |
| Simulator | MuJoCo via `robosuite` | PyBullet | CoppeliaSim |
| What's held fixed to isolate a variable | Objects/goal (Spatial), layout/goal (Object), or objects/layout (Goal) — one axis at a time | Scene and skill vocabulary shared across all instruction chains | Varies per task; less designed around controlled shift |
| Headline evaluation protocol | Sequential training over tasks + rollout success matrix $S_{k,j}$; FWT/NBT/AUC | Rolling success across chained instructions (how many consecutive sub-goals completed) | Per-task success rate, often single-task trained |

The thing worth internalizing for your own notebook-writing and, eventually, research judgment: **CALVIN and RLBench are both, in a sense, testing "can a policy do many things," while LIBERO is testing "can a policy learn many things one at a time without forgetting the earlier ones."** That's a different axis, and it's why LIBERO's contribution isn't "more tasks" — LIBERO-Object and LIBERO-Goal individually have *fewer* tasks than a single RLBench category — but a different, controlled *evaluation protocol* layered on top of tasks. If a future paper you read claims a "LIBERO result," check whether it's reporting single-task success (comparable to RLBench-style numbers) or the actual lifelong FWT/NBT/AUC protocol — the two are easy to conflate and mean very different things.

## 17. Discussion questions

Use these to prep for group meeting — the goal is to be able to explain *why* the benchmark is designed the way it is and defend/attack its results, not just recite numbers.

**Design & scope**

1. Why does LIBERO deliberately separate declarative and procedural knowledge shift into different suites (Spatial / Object / Goal), instead of one big diverse task pool? What would we lose if we could only measure aggregate performance on LIBERO-100?
2. LIBERO shares the same observation space, action space, and (in principle) the same network architecture across all 130 tasks. Why does that matter specifically for studying *transfer*, as opposed to just having a larger benchmark?
3. Why is it important that `get_task_init_states` returns a **fixed** pool of initial states rather than random resets on every evaluation? What happens to comparability across papers if each reimplementation resets differently?

**Results & interpretation**

4. SeqL beats purpose-built lifelong algorithms (EWC, PackNet, ER) on forward transfer. Does that mean those algorithms are bad? Which metric would you expect them to win on instead, and why might none of them clearly win on AUC either?
5. EWC underperforms plain sequential fine-tuning. Walk through the EWC penalty term in Section 5.3 and explain, mechanically, how a large Fisher-weighted penalty could hurt a *new* task's performance even if it perfectly protects old ones.
6. PackNet struggles specifically on LIBERO-10/LIBERO-Long. Given how PackNet works (freeze-and-prune per task), propose a concrete failure mode that would explain why longer-horizon tasks are disproportionately affected.
7. The paper finds naive supervised pretraining on LIBERO-90 can hurt LIBERO-10 lifelong performance. Propose one hypothesis for *why*, and one experiment — using tools already in this notebook (the metrics, the benchmark structure, the demo data) — that would test it.
8. Pretrained language embeddings (BERT/CLIP/GPT-2) don't clearly beat a bag-of-words-level Task-ID embedding. If you wanted to design an experiment to test whether a policy is actually using compositional language understanding versus just task discrimination, what would you change about the evaluation (hint: think about instructions the policy has never seen verbatim)?

**Extending the benchmark**

9. LIBERO uses a sparse reward (+1 on completion) and states this makes pure RL "extremely hard," pushing the benchmark toward imitation learning. What would you need to add (reward shaping, curriculum, something else) to make LIBERO tractable for RL without human demonstrations?
10. If you were designing a "LIBERO-5" — a fifth suite isolating a *new* axis of knowledge transfer not covered by Spatial/Object/Goal — what axis would you pick, and what would you have to hold fixed to isolate it cleanly?

## References

- Liu, B., Zhu, Y., Gao, C., Feng, Y., Liu, Q., Zhu, Y., Stone, P. (2023). *LIBERO: Benchmarking Knowledge Transfer for Lifelong Robot Learning.* arXiv:2306.03310 / NeurIPS 2023 Datasets & Benchmarks.
- Official repository: [Lifelong-Robot-Learning/LIBERO](https://github.com/Lifelong-Robot-Learning/LIBERO)
- Documentation: [lifelong-robot-learning.github.io/LIBERO](https://lifelong-robot-learning.github.io/LIBERO/)
- Project website & leaderboard: [libero-project.github.io](https://libero-project.github.io)
- `robosuite` simulation framework: [robosuite.ai](https://robosuite.ai/)
- Lopez-Paz, D. & Ranzato, M. (2017). *Gradient Episodic Memory for Continual Learning.* NeurIPS — source of the forward/backward transfer metric convention LIBERO adapts.
- Kirkpatrick, J. et al. (2017). *Overcoming catastrophic forgetting in neural networks* (EWC). PNAS.
- Mallya, A. & Lazebnik, S. (2018). *PackNet: Adding Multiple Tasks to a Single Network by Iterative Pruning.* CVPR.
- Mees, O. et al. (2022). *CALVIN: A Benchmark for Language-Conditioned Policy Learning for Long-Horizon Robot Manipulation Tasks.* — related benchmark, see Section 16 comparison.
- James, S. et al. (2020). *RLBench: The Robot Learning Benchmark & Learning Environment.* — related benchmark, see Section 16 comparison.